# 01. EDA & Preprocessing Verification

## Overview
This notebook verifies the data loading and preprocessing pipeline for the Diabetic Retinopathy Detection project.

## Objectives
1. **Manifest Creation**: Parse `trainLabels.csv` and ensure image files match.
2. **Distribution Analysis**: Visualize the class imbalance (0-4 scales).
3. **Preprocessing Check**: Verify the `crop_black_borders` and resize logic eliminates artifacts without destroying the fundus.
4. **Quality Assessment**: Evaluate image quality using **Vessel Segmentation Density** to identify low-contrast or blurry images.

In [ ]:
# --- Imports ---
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Add project root to path
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import load_config, DATA_DIR, PROCESSED_DATA_DIR
from src.data.preprocessing import ImagePreprocessor
from src.data import RetinopathyDataset
from src.data.make_dataset import create_manifest
from src.data.transforms import IsotropicAffineTransform
from src.visualization.plots import set_plot_style, plot_class_distribution
from src.visualization.eda import (
    load_and_show_preprocessing,
    show_vessel_density_report,
    show_augmentation_verification,
)

# --- Configuration ---
set_plot_style()
config = load_config()
print(f"Data Dir: {DATA_DIR}")

## 2. Load and Inspect Manifest
We load the labels and create a unified manifest.

In [ ]:
manifest_path = PROCESSED_DATA_DIR / "manifest.csv"

# Create or Load Manifest
try:
    if manifest_path.exists():
        print("Loading existing manifest...")
        df = pd.read_csv(manifest_path)
    else:
        print("Creating new manifest...")
        df = create_manifest(DATA_DIR, manifest_path)
except Exception as e:
    # Fallback if logic fails (e.g. files missing) - load csv directly for demo
    print(f"Manifest creation warning: {e}")
    df = pd.read_csv(DATA_DIR / "trainLabels.csv")
    # patch paths for demo
    df['path'] = df['image'].apply(lambda x: str(DATA_DIR / "train" / (x + ".jpeg")))

print(f"Total Samples: {len(df)}")
df.head()

## 3. Class Distribution
The dataset is known to be heavily imbalanced.

In [ ]:
fig = plot_class_distribution(df, label_col='level', title="Diabetic Retinopathy Grade Distribution")
plt.show()

## 4. Preprocessing Verification
We test the `ImagePreprocessor` on a random batch of images.
We want to see the effect of cropping black borders.

In [ ]:
# 4. Preprocessing Verification
preprocessor = ImagePreprocessor(target_size=(512, 512), sigmaX=30)

samples = load_and_show_preprocessing(df, preprocessor, n_samples=6, random_state=42)

### Preprocessing Observations
- **Successes**: The pipeline successfully standardizes retinal scale (zoom) and removes confounding color variations (via Graham's method), effectively highlighting structural patterns like vessels and lesions.
- **Limitations**: The orientation standardization (flipping left eyes to look like right eyes) shows inconsistencies. This appears to be due to inherent mislabeling of eye side in the raw dataset (e.g., a "right" label on a physically "left" image).
  - *Mitigation*: This is a known issue with EyePACS and should not negatively impact the network, as we will use random horizontal flips during training which makes the model invariant to laterality.

## 5. Quality Assessment (QC)
We replace the simple Blur Score with **Vessel Segmentation Density**.
- **Logic**: A high-quality fundus image must have visible, sharp blood vessels.
- **Metric**: Percentage of the retina area covered by detected vessels (Adaptive Thresholding).
- **Visualization**: We overlay the detected vessel map (Red) to confirm the metric aligns with visual reality.

In [ ]:
# 5. Vessel Density Quality Check
qc_df = show_vessel_density_report(samples, preprocessor)

## 6. Dataset Class Test
This cell verifies that the PyTorch Dataset correctly uses our Preprocessor.


In [ ]:
# 6. Dataset Class Test
print("Verifying Dataset Integration...")

# 1. Initialize Preprocessor (Exact same config as above)
prod_preprocessor = ImagePreprocessor(target_size=(512, 512), sigmaX=30)

# 2. Create Dataset Instance
dataset_check = RetinopathyDataset(df=df, preprocessor=prod_preprocessor)

# 3. Retrieve a Single Sample (Simulate what the DataLoader does)
sample_idx = 0
data_item = dataset_check[sample_idx]

tensor_img = data_item['image']
label = data_item['label']

print(f"\n--- Dataset Output for Index {sample_idx} ---")
print(f"Tensor Shape: {tensor_img.shape} (Channels, Height, Width)")
print(f"Value Range:  [{tensor_img.min():.4f}, {tensor_img.max():.4f}]")
print(f"Data Type:    {tensor_img.dtype}")
print(f"Label:        {label}")

# 4. Visualize the Tensor (C,H,W -> H,W,C)
plt.figure(figsize=(6, 6))
plt.imshow(tensor_img.permute(1, 2, 0).numpy())
plt.title(f"Dataset Class Output\nLabel: {label}")
plt.axis('off')
plt.show()

## 7. Data Augmentation Verification
We verify the **Albumentations** pipeline. 
Since medical images can be rotated freely (the retina is circular), we apply aggressive geometric transformations (Rotation, Flips) to prevent overfitting.
We also verify that the `RetinopathyDataset` correctly hooks into these transforms.

In [ ]:
# 7. Data Augmentation Verification

# Define Aggressive Pipeline for Demo
iso_transforms = A.Compose([
    IsotropicAffineTransform(scale_limit=0.4, shift_limit=0.2, rotate_limit=180, p=1.0),
    ToTensorV2()
])

# Create Dataset
iso_dataset = RetinopathyDataset(df=df, preprocessor=prod_preprocessor, transforms=iso_transforms)

# Visualize Augmentations
sample_row = df.iloc[0]
show_augmentation_verification(
    iso_dataset, sample_idx=0, n_augments=12,
    patient_id=str(sample_row['patient_id'])
)